In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("../data/processed/telco_cleaned.csv")

df.head()

,CustomerID,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,7892-POOKP,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


In [3]:
target = "CLTV"

y = df[target]

In [4]:
df.columns.tolist()

['CustomerID',
 'City',
 'Zip Code',
 'Lat Long',
 'Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Churn Label',
 'Churn Value',
 'Churn Score',
 'CLTV',
 'Churn Reason']

In [5]:
df[target].describe()

count    7043.000000
mean     4400.295755
std      1183.057152
min      2003.000000
25%      3469.000000
50%      4527.000000
75%      5380.500000
max      6500.000000
Name: CLTV, dtype: float64

In [6]:
df["Tenure Group"] = pd.cut(
    df["Tenure Months"],
    bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)


In [7]:
drop_cols = [
    "CustomerID",
    "CLTV",
    "Churn Label",
    "Churn Value",
    "Churn Score",
    "Churn Reason",
    "City",
    "Lat Long",
    "Zip Code"
]

X = df.drop(columns=drop_cols)

In [8]:
y = df["CLTV"]

In [9]:
X.columns.tolist()

['Latitude',
 'Longitude',
 'Gender',
 'Senior Citizen',
 'Partner',
 'Dependents',
 'Tenure Months',
 'Phone Service',
 'Multiple Lines',
 'Internet Service',
 'Online Security',
 'Online Backup',
 'Device Protection',
 'Tech Support',
 'Streaming TV',
 'Streaming Movies',
 'Contract',
 'Paperless Billing',
 'Payment Method',
 'Monthly Charges',
 'Total Charges',
 'Tenure Group']

In [10]:
X.shape

(7043, 22)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [12]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (5634, 22)
X_test : (1409, 22)
y_train: (5634,)
y_test : (1409,)


In [13]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["int64", "float64"]
).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges']

Categorical features:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Tenure Group']


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [15]:
models = {
    "Linear Regression": LinearRegression(),
    
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),
    
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1
    )
}

In [16]:
results = []

for name, regressor in models.items():

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", regressor)
        ]
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R²": r2_score(y_test, y_pred)
    })

In [17]:
comparison = pd.DataFrame(results)

comparison.round(3)

,Model,MAE,RMSE,R²
0,Linear Regression,872.315,1026.241,0.225
1,Random Forest,893.701,1055.757,0.179
2,XGBoost,882.510,1040.645,0.203


In [18]:
correlation = (
    df.select_dtypes(include=["int64", "float64"])
      .corr()["CLTV"]
      .sort_values(ascending=False)
)

correlation

CLTV               1.000000
Tenure Months      0.396406
Total Charges      0.342091
Monthly Charges    0.098693
Latitude           0.000886
Longitude          0.000485
Zip Code          -0.003562
Churn Score       -0.079782
Churn Value       -0.127463
Name: CLTV, dtype: float64

In [19]:
final_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

cv_scores = cross_val_score(
    final_model,
    X,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("CV R² scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())
print("Std CV R²:", cv_scores.std())

CV R² scores: [0.09353655 0.22191945 0.25331537 0.23402747 0.25679422]
Mean CV R²: 0.21191861028166584
Std CV R²: 0.060548562335748915


In [20]:
import joblib

final_model.fit(X, y)

# Save model
joblib.dump(
    final_model,
    "../models/linear_regression_cltv_model.pkl"
)

# Save feature names
joblib.dump(
    X.columns.tolist(),
    "../models/cltv_features.pkl"
)

print("Final CLTV model saved successfully.")

Final CLTV model saved successfully.


# Regression Summary

## Objective
Predict Customer Lifetime Value (CLTV).

## Target
- `CLTV`

## Data Preparation
- Constant and identifier columns were excluded.
- Target-related and post-outcome columns were excluded to avoid leakage.
- Numerical features were standardized.
- Categorical features were one-hot encoded.

## Feature Engineering
- `Tenure Group` was retained because it improved model performance.
- `Charge_Per_Tenure_Ratio` was retained, although its improvement was small.
- Other tested engineered features were removed because they did not provide meaningful improvement.

## Models Evaluated
- Linear Regression
- Random Forest Regressor
- XGBoost Regressor

## Final Model
**Linear Regression**

### Holdout Test Performance
- MAE: **871.97**
- RMSE: **1025.55**
- R²: **0.226**

### 5-Fold Cross-Validation
- Mean R²: **0.212**
- Standard Deviation: **0.061**

The cross-validation result was broadly consistent with the holdout test performance, although some variation was observed across folds.

## Final Decision
Linear Regression was selected as the final CLTV regression model based on its performance on the final feature set.

The final pipeline was trained on the full dataset and saved as:

`models/linear_regression_cltv_model.pkl`